# Employee Attrition Prediction — Data Cleaning

This notebook cleans and validates the IBM HR Analytics Employee Attrition dataset before preprocessing and model training.

## Objectives

- Load the raw dataset safely
- Remove duplicate rows
- Remove constant columns
- Remove identifier columns
- Validate categorical values
- Validate numerical ranges and logical relationships
- Convert the target variable into binary values
- Save the cleaned dataset and cleaning reports

In [1]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 140)

print("Libraries imported successfully.")

Libraries imported successfully.


## 1. Configure Project Paths

Expected project structure:

```text
employee-attrition-prediction/
├── data/
│   ├── raw/
│   │   └── WA_Fn-UseC_-HR-Employee-Attrition.csv
│   └── processed/
├── notebooks/
│   └── 02_data_cleaning.ipynb
└── reports/
```

In [2]:
CURRENT_DIR = Path.cwd()

PROJECT_ROOT = (
    CURRENT_DIR.parent
    if CURRENT_DIR.name == "notebooks"
    else CURRENT_DIR
)

RAW_DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "WA_Fn-UseC_-HR-Employee-Attrition.csv"
)

PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports"

CLEAN_DATA_PATH = (
    PROCESSED_DATA_DIR
    / "clean_employee_attrition.csv"
)

CLEANING_SUMMARY_PATH = (
    REPORTS_DIR
    / "cleaning_summary.json"
)

REMOVED_COLUMNS_PATH = (
    REPORTS_DIR
    / "removed_columns.txt"
)

VALIDATION_REPORT_PATH = (
    REPORTS_DIR
    / "data_validation_report.csv"
)

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT.resolve())
print("Raw dataset:", RAW_DATA_PATH.resolve())
print("Clean dataset output:", CLEAN_DATA_PATH.resolve())

Project root: C:\Users\Nasteho Abdi\employee-attrition-prediction
Raw dataset: C:\Users\Nasteho Abdi\employee-attrition-prediction\data\raw\WA_Fn-UseC_-HR-Employee-Attrition.csv
Clean dataset output: C:\Users\Nasteho Abdi\employee-attrition-prediction\data\processed\clean_employee_attrition.csv


## 2. Load the Raw Dataset

In [3]:
if not RAW_DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at: {RAW_DATA_PATH.resolve()}\n"
        "Place WA_Fn-UseC_-HR-Employee-Attrition.csv inside data/raw/."
    )

raw_df = pd.read_csv(RAW_DATA_PATH)

print("Dataset loaded successfully.")
print("Original shape:", raw_df.shape)

raw_df.head()

Dataset loaded successfully.
Original shape: (1470, 35)


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,EnvironmentSatisfaction,Gender,HourlyRate,JobInvolvement,JobLevel,JobRole,JobSatisfaction,MaritalStatus,MonthlyIncome,MonthlyRate,NumCompaniesWorked,Over18,OverTime,PercentSalaryHike,PerformanceRating,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,2,Female,94,3,2,Sales Executive,4,Single,5993,19479,8,Y,Yes,11,3,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,3,Male,61,2,2,Research Scientist,2,Married,5130,24907,1,Y,No,23,4,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,4,Male,92,2,1,Laboratory Technician,3,Single,2090,2396,6,Y,Yes,15,3,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,4,Female,56,3,1,Research Scientist,3,Married,2909,23159,1,Y,Yes,11,3,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,1,Male,40,3,1,Laboratory Technician,2,Married,3468,16632,9,Y,No,12,3,4,80,1,6,3,3,2,2,2,2


## 3. Create a Working Copy

The raw dataset remains unchanged. All cleaning operations are applied to a separate copy.

In [4]:
clean_df = raw_df.copy()

original_rows = int(clean_df.shape[0])
original_columns = int(clean_df.shape[1])

print("Working copy created.")

Working copy created.


## 4. Standardize Column Names

The IBM dataset already has consistent column names. This step removes accidental spaces from column names without changing their meaning.

In [5]:
original_column_names = clean_df.columns.tolist()

clean_df.columns = clean_df.columns.str.strip()

renamed_columns = {
    old: new
    for old, new in zip(original_column_names, clean_df.columns)
    if old != new
}

print("Renamed columns:", renamed_columns)

Renamed columns: {}


## 5. Inspect Missing Values

In [6]:
missing_before = clean_df.isnull().sum()

missing_report_before = pd.DataFrame({
    "Column": clean_df.columns,
    "Missing Count": missing_before.values,
    "Missing Percentage": (
        clean_df.isnull().mean().mul(100).round(2).values
    ),
})

print("Total missing values:", int(missing_before.sum()))

missing_report_before[
    missing_report_before["Missing Count"] > 0
]

Total missing values: 0


,Column,Missing Count,Missing Percentage


The original IBM dataset normally has no missing values. We do not invent replacement values in this cleaning phase. Any missing-value handling required later will be placed inside the machine-learning preprocessing pipeline to avoid data leakage.

## 6. Remove Duplicate Rows

In [7]:
duplicate_rows_before = int(clean_df.duplicated().sum())

print("Duplicate rows before cleaning:", duplicate_rows_before)

Duplicate rows before cleaning: 0


In [8]:
if duplicate_rows_before > 0:
    clean_df = clean_df.drop_duplicates().reset_index(drop=True)
    print(f"Removed {duplicate_rows_before} duplicate rows.")
else:
    print("No duplicate rows were found.")

print("Shape after duplicate removal:", clean_df.shape)

No duplicate rows were found.
Shape after duplicate removal: (1470, 35)


## 7. Detect and Remove Constant Columns

A constant column contains only one unique value. It cannot help a model distinguish between employees.

In [9]:
constant_columns = [
    column
    for column in clean_df.columns
    if clean_df[column].nunique(dropna=False) == 1
]

print("Constant columns:", constant_columns)

Constant columns: ['EmployeeCount', 'Over18', 'StandardHours']


In [10]:
if constant_columns:
    clean_df = clean_df.drop(columns=constant_columns)

print("Shape after constant-column removal:", clean_df.shape)

Shape after constant-column removal: (1470, 32)


The expected constant columns in this dataset are usually:

- `EmployeeCount`
- `Over18`
- `StandardHours`

## 8. Detect and Remove Identifier Columns

`EmployeeNumber` uniquely identifies each employee. It is not a meaningful behavioral feature and can cause the model to learn accidental patterns.

In [11]:
candidate_identifier_columns = [
    column
    for column in clean_df.columns
    if clean_df[column].nunique(dropna=False) == len(clean_df)
]

print("Columns with unique values for every row:")
print(candidate_identifier_columns)

Columns with unique values for every row:
['EmployeeNumber']


In [12]:
identifier_columns = [
    column
    for column in ["EmployeeNumber"]
    if column in clean_df.columns
]

if identifier_columns:
    clean_df = clean_df.drop(columns=identifier_columns)

print("Removed identifier columns:", identifier_columns)
print("Shape after identifier removal:", clean_df.shape)

Removed identifier columns: ['EmployeeNumber']
Shape after identifier removal: (1470, 31)


## 9. Clean String Columns

Leading and trailing whitespace can create fake categories such as `"Sales"` and `"Sales "`.

In [13]:
string_columns = clean_df.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

for column in string_columns:
    clean_df[column] = clean_df[column].apply(
        lambda value: value.strip()
        if isinstance(value, str)
        else value
    )

print("String columns cleaned:", string_columns)

String columns cleaned: ['Attrition', 'BusinessTravel', 'Department', 'EducationField', 'Gender', 'JobRole', 'MaritalStatus', 'OverTime']


## 10. Validate the Target Column

In [14]:
TARGET_COLUMN = "Attrition"

if TARGET_COLUMN not in clean_df.columns:
    raise ValueError(
        f"Target column '{TARGET_COLUMN}' is missing."
    )

target_values_before = sorted(
    clean_df[TARGET_COLUMN]
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)

print("Target values before conversion:", target_values_before)

expected_target_values = {"Yes", "No"}
unexpected_target_values = (
    set(target_values_before) - expected_target_values
)

if unexpected_target_values:
    raise ValueError(
        "Unexpected target values found: "
        f"{sorted(unexpected_target_values)}"
    )

Target values before conversion: ['No', 'Yes']


## 11. Convert Attrition to Binary Values

- `No` becomes `0`
- `Yes` becomes `1`

In [15]:
target_mapping = {
    "No": 0,
    "Yes": 1,
}

clean_df[TARGET_COLUMN] = (
    clean_df[TARGET_COLUMN]
    .map(target_mapping)
)

if clean_df[TARGET_COLUMN].isnull().any():
    raise ValueError(
        "Target conversion created missing values. "
        "Inspect the original target labels."
    )

clean_df[TARGET_COLUMN] = (
    clean_df[TARGET_COLUMN]
    .astype("int64")
)

clean_df[TARGET_COLUMN].value_counts().sort_index()

Attrition
0    1233
1     237
Name: count, dtype: int64

## 12. Inspect Categorical Values

In [16]:
categorical_columns = clean_df.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

categorical_value_report = {}

for column in categorical_columns:
    values = sorted(
        clean_df[column]
        .dropna()
        .astype(str)
        .unique()
        .tolist()
    )

    categorical_value_report[column] = values

    print("=" * 70)
    print(f"{column} ({len(values)} unique values)")
    print(values)

BusinessTravel (3 unique values)
['Non-Travel', 'Travel_Frequently', 'Travel_Rarely']
Department (3 unique values)
['Human Resources', 'Research & Development', 'Sales']
EducationField (6 unique values)
['Human Resources', 'Life Sciences', 'Marketing', 'Medical', 'Other', 'Technical Degree']
Gender (2 unique values)
['Female', 'Male']
JobRole (9 unique values)
['Healthcare Representative', 'Human Resources', 'Laboratory Technician', 'Manager', 'Manufacturing Director', 'Research Director', 'Research Scientist', 'Sales Executive', 'Sales Representative']
MaritalStatus (3 unique values)
['Divorced', 'Married', 'Single']
OverTime (2 unique values)
['No', 'Yes']


## 13. Validate Known Categorical Domains

This section checks important columns against the categories expected in the IBM dataset. It reports unexpected values instead of silently deleting rows.

In [17]:
expected_categories = {
    "BusinessTravel": {
        "Non-Travel",
        "Travel_Frequently",
        "Travel_Rarely",
    },
    "Department": {
        "Human Resources",
        "Research & Development",
        "Sales",
    },
    "Gender": {
        "Female",
        "Male",
    },
    "MaritalStatus": {
        "Divorced",
        "Married",
        "Single",
    },
    "OverTime": {
        "No",
        "Yes",
    },
}

categorical_validation = []

for column, expected_values in expected_categories.items():
    if column not in clean_df.columns:
        categorical_validation.append({
            "Column": column,
            "Check": "Expected categories",
            "Status": "Column missing",
            "Details": "",
        })
        continue

    actual_values = set(
        clean_df[column]
        .dropna()
        .astype(str)
        .unique()
    )

    unexpected_values = sorted(
        actual_values - expected_values
    )

    categorical_validation.append({
        "Column": column,
        "Check": "Expected categories",
        "Status": (
            "Pass"
            if not unexpected_values
            else "Review"
        ),
        "Details": ", ".join(unexpected_values),
    })

categorical_validation_df = pd.DataFrame(
    categorical_validation
)

categorical_validation_df

,Column,Check,Status,Details
0,BusinessTravel,Expected categories,Pass,
1,Department,Expected categories,Pass,
2,Gender,Expected categories,Pass,
3,MaritalStatus,Expected categories,Pass,
4,OverTime,Expected categories,Pass,


## 14. Inspect Numerical Features

In [18]:
numerical_columns = clean_df.select_dtypes(
    include=np.number
).columns.tolist()

clean_df[numerical_columns].describe().T

,count,mean,std,min,25%,50%,75%,max
Age,1470.0,36.923810,9.135373,18.0,30.0,36.0,43.00,60.0
Attrition,1470.0,0.161224,0.367863,0.0,0.0,0.0,0.00,1.0
DailyRate,1470.0,802.485714,403.509100,102.0,465.0,802.0,1157.00,1499.0
DistanceFromHome,1470.0,9.192517,8.106864,1.0,2.0,7.0,14.00,29.0
Education,1470.0,2.912925,1.024165,1.0,2.0,3.0,4.00,5.0
EnvironmentSatisfaction,1470.0,2.721769,1.093082,1.0,2.0,3.0,4.00,4.0
HourlyRate,1470.0,65.891156,20.329428,30.0,48.0,66.0,83.75,100.0
JobInvolvement,1470.0,2.729932,0.711561,1.0,2.0,3.0,3.00,4.0
JobLevel,1470.0,2.063946,1.106940,1.0,1.0,2.0,3.00,5.0
JobSatisfaction,1470.0,2.728571,1.102846,1.0,2.0,3.0,4.00,4.0


## 15. Check for Negative Numerical Values

Most numerical features in this HR dataset should not be negative.

In [19]:
negative_value_report = []

for column in numerical_columns:
    negative_count = int(
        (clean_df[column] < 0).sum()
    )

    if negative_count > 0:
        negative_value_report.append({
            "Column": column,
            "Negative Values": negative_count,
        })

negative_value_df = pd.DataFrame(
    negative_value_report
)

if negative_value_df.empty:
    print("No negative numerical values were found.")
else:
    display(negative_value_df)

No negative numerical values were found.


## 16. Validate Important Numerical Ranges

These checks are based on the known structure of the IBM dataset and general logical constraints.

In [20]:
range_rules = {
    "Age": (18, 100),
    "DistanceFromHome": (0, None),
    "Education": (1, 5),
    "EnvironmentSatisfaction": (1, 4),
    "JobInvolvement": (1, 4),
    "JobLevel": (1, 5),
    "JobSatisfaction": (1, 4),
    "MonthlyIncome": (0, None),
    "NumCompaniesWorked": (0, None),
    "PercentSalaryHike": (0, 100),
    "PerformanceRating": (1, 4),
    "RelationshipSatisfaction": (1, 4),
    "StockOptionLevel": (0, None),
    "TotalWorkingYears": (0, None),
    "TrainingTimesLastYear": (0, None),
    "WorkLifeBalance": (1, 4),
    "YearsAtCompany": (0, None),
    "YearsInCurrentRole": (0, None),
    "YearsSinceLastPromotion": (0, None),
    "YearsWithCurrManager": (0, None),
}

range_validation = []

for column, (minimum, maximum) in range_rules.items():
    if column not in clean_df.columns:
        continue

    invalid_mask = pd.Series(False, index=clean_df.index)

    if minimum is not None:
        invalid_mask |= clean_df[column] < minimum

    if maximum is not None:
        invalid_mask |= clean_df[column] > maximum

    invalid_count = int(invalid_mask.sum())

    range_validation.append({
        "Column": column,
        "Check": f"Range [{minimum}, {maximum}]",
        "Status": "Pass" if invalid_count == 0 else "Review",
        "Invalid Rows": invalid_count,
    })

range_validation_df = pd.DataFrame(
    range_validation
)

range_validation_df

,Column,Check,Status,Invalid Rows
0,Age,"Range [18, 100]",Pass,0
1,DistanceFromHome,"Range [0, None]",Pass,0
2,Education,"Range [1, 5]",Pass,0
3,EnvironmentSatisfaction,"Range [1, 4]",Pass,0
4,JobInvolvement,"Range [1, 4]",Pass,0
5,JobLevel,"Range [1, 5]",Pass,0
6,JobSatisfaction,"Range [1, 4]",Pass,0
7,MonthlyIncome,"Range [0, None]",Pass,0
8,NumCompaniesWorked,"Range [0, None]",Pass,0
9,PercentSalaryHike,"Range [0, 100]",Pass,0


## 17. Validate Logical Relationships Between Work-Experience Columns

These relationships should normally hold:

- `YearsAtCompany` ≤ `TotalWorkingYears`
- `YearsInCurrentRole` ≤ `YearsAtCompany`
- `YearsSinceLastPromotion` ≤ `YearsAtCompany`
- `YearsWithCurrManager` ≤ `YearsAtCompany`

In [21]:
logical_rules = []

relationship_checks = [
    (
        "YearsAtCompany <= TotalWorkingYears",
        "YearsAtCompany",
        "TotalWorkingYears",
    ),
    (
        "YearsInCurrentRole <= YearsAtCompany",
        "YearsInCurrentRole",
        "YearsAtCompany",
    ),
    (
        "YearsSinceLastPromotion <= YearsAtCompany",
        "YearsSinceLastPromotion",
        "YearsAtCompany",
    ),
    (
        "YearsWithCurrManager <= YearsAtCompany",
        "YearsWithCurrManager",
        "YearsAtCompany",
    ),
]

for rule_name, left_column, right_column in relationship_checks:
    if (
        left_column not in clean_df.columns
        or right_column not in clean_df.columns
    ):
        continue

    invalid_count = int(
        (
            clean_df[left_column]
            > clean_df[right_column]
        ).sum()
    )

    logical_rules.append({
        "Rule": rule_name,
        "Status": "Pass" if invalid_count == 0 else "Review",
        "Invalid Rows": invalid_count,
    })

logical_validation_df = pd.DataFrame(
    logical_rules
)

logical_validation_df

,Rule,Status,Invalid Rows
0,YearsAtCompany <= TotalWorkingYears,Pass,0
1,YearsInCurrentRole <= YearsAtCompany,Pass,0
2,YearsSinceLastPromotion <= YearsAtCompany,Pass,0
3,YearsWithCurrManager <= YearsAtCompany,Pass,0


Invalid values are reported rather than automatically removed. Automatic row deletion could discard legitimate records or hide a data-quality problem. Any issue marked `Review` should be examined manually.

## 18. Check Missing Values After Cleaning

In [22]:
missing_after = int(
    clean_df.isnull().sum().sum()
)

print("Missing values after cleaning:", missing_after)

Missing values after cleaning: 0


## 19. Check Duplicate Rows After Cleaning

In [23]:
duplicates_after = int(
    clean_df.duplicated().sum()
)

print("Duplicate rows after cleaning:", duplicates_after)

Duplicate rows after cleaning: 0


## 20. Validate Final Target Distribution

In [24]:
target_distribution = pd.DataFrame({
    "Count": clean_df[TARGET_COLUMN]
    .value_counts()
    .sort_index(),
    "Percentage": clean_df[TARGET_COLUMN]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2),
})

target_distribution.index = [
    "Stay (0)",
    "Leave (1)",
]

target_distribution

,Count,Percentage
Stay (0),1233,83.88
Leave (1),237,16.12


## 21. Inspect Final Dataset

In [25]:
print("Original shape:", raw_df.shape)
print("Clean shape:", clean_df.shape)

clean_df.head()

Original shape: (1470, 35)
Clean shape: (1470, 31)


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EnvironmentSatisfaction,Gender,HourlyRate,JobInvolvement,JobLevel,JobRole,JobSatisfaction,MaritalStatus,MonthlyIncome,MonthlyRate,NumCompaniesWorked,OverTime,PercentSalaryHike,PerformanceRating,RelationshipSatisfaction,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,1,Travel_Rarely,1102,Sales,1,2,Life Sciences,2,Female,94,3,2,Sales Executive,4,Single,5993,19479,8,Yes,11,3,1,0,8,0,1,6,4,0,5
1,49,0,Travel_Frequently,279,Research & Development,8,1,Life Sciences,3,Male,61,2,2,Research Scientist,2,Married,5130,24907,1,No,23,4,4,1,10,3,3,10,7,1,7
2,37,1,Travel_Rarely,1373,Research & Development,2,2,Other,4,Male,92,2,1,Laboratory Technician,3,Single,2090,2396,6,Yes,15,3,2,0,7,3,3,0,0,0,0
3,33,0,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,4,Female,56,3,1,Research Scientist,3,Married,2909,23159,1,Yes,11,3,3,0,8,3,3,8,7,3,0
4,27,0,Travel_Rarely,591,Research & Development,2,1,Medical,1,Male,40,3,1,Laboratory Technician,2,Married,3468,16632,9,No,12,3,4,1,6,3,3,2,2,2,2


In [26]:
clean_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1470 entries, 0 to 1469
Data columns (total 31 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   Age                       1470 non-null   int64
 1   Attrition                 1470 non-null   int64
 2   BusinessTravel            1470 non-null   str  
 3   DailyRate                 1470 non-null   int64
 4   Department                1470 non-null   str  
 5   DistanceFromHome          1470 non-null   int64
 6   Education                 1470 non-null   int64
 7   EducationField            1470 non-null   str  
 8   EnvironmentSatisfaction   1470 non-null   int64
 9   Gender                    1470 non-null   str  
 10  HourlyRate                1470 non-null   int64
 11  JobInvolvement            1470 non-null   int64
 12  JobLevel                  1470 non-null   int64
 13  JobRole                   1470 non-null   str  
 14  JobSatisfaction           1470 non-null   int64
 15

## 22. Build a Combined Validation Report

In [27]:
validation_frames = []

if not categorical_validation_df.empty:
    validation_frames.append(
        categorical_validation_df.rename(
            columns={"Details": "Result"}
        )[["Column", "Check", "Status", "Result"]]
    )

if not range_validation_df.empty:
    range_report = range_validation_df.copy()
    range_report["Result"] = (
        range_report["Invalid Rows"]
        .astype(str)
        + " invalid rows"
    )
    validation_frames.append(
        range_report[
            ["Column", "Check", "Status", "Result"]
        ]
    )

if not logical_validation_df.empty:
    logical_report = logical_validation_df.copy()
    logical_report["Column"] = "Multiple columns"
    logical_report["Check"] = logical_report["Rule"]
    logical_report["Result"] = (
        logical_report["Invalid Rows"]
        .astype(str)
        + " invalid rows"
    )
    validation_frames.append(
        logical_report[
            ["Column", "Check", "Status", "Result"]
        ]
    )

if validation_frames:
    validation_report = pd.concat(
        validation_frames,
        ignore_index=True,
    )
else:
    validation_report = pd.DataFrame(
        columns=["Column", "Check", "Status", "Result"]
    )

validation_report

,Column,Check,Status,Result
0,BusinessTravel,Expected categories,Pass,
1,Department,Expected categories,Pass,
2,Gender,Expected categories,Pass,
3,MaritalStatus,Expected categories,Pass,
4,OverTime,Expected categories,Pass,
5,Age,"Range [18, 100]",Pass,0 invalid rows
6,DistanceFromHome,"Range [0, None]",Pass,0 invalid rows
7,Education,"Range [1, 5]",Pass,0 invalid rows
8,EnvironmentSatisfaction,"Range [1, 4]",Pass,0 invalid rows
9,JobInvolvement,"Range [1, 4]",Pass,0 invalid rows


## 23. Save the Cleaned Dataset

In [28]:
clean_df.to_csv(
    CLEAN_DATA_PATH,
    index=False,
)

print(
    "Clean dataset saved successfully to:",
    CLEAN_DATA_PATH.resolve(),
)

Clean dataset saved successfully to: C:\Users\Nasteho Abdi\employee-attrition-prediction\data\processed\clean_employee_attrition.csv


## 24. Save Cleaning Reports

In [29]:
removed_rows = original_rows - int(clean_df.shape[0])
removed_column_names = (
    constant_columns
    + identifier_columns
)

cleaning_summary = {
    "source_file": RAW_DATA_PATH.name,
    "output_file": CLEAN_DATA_PATH.name,
    "original_rows": original_rows,
    "clean_rows": int(clean_df.shape[0]),
    "rows_removed": removed_rows,
    "duplicate_rows_removed": duplicate_rows_before,
    "original_columns": original_columns,
    "clean_columns": int(clean_df.shape[1]),
    "columns_removed_count": len(removed_column_names),
    "constant_columns_removed": constant_columns,
    "identifier_columns_removed": identifier_columns,
    "renamed_columns": renamed_columns,
    "missing_values_before": int(
        raw_df.isnull().sum().sum()
    ),
    "missing_values_after": missing_after,
    "duplicates_after_cleaning": duplicates_after,
    "target_mapping": {
        "No": 0,
        "Yes": 1,
    },
    "target_class_counts": {
        "stay_0": int(
            (clean_df[TARGET_COLUMN] == 0).sum()
        ),
        "leave_1": int(
            (clean_df[TARGET_COLUMN] == 1).sum()
        ),
    },
}

with CLEANING_SUMMARY_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        cleaning_summary,
        file,
        indent=4,
    )

print(
    "Cleaning summary saved to:",
    CLEANING_SUMMARY_PATH.resolve(),
)

Cleaning summary saved to: C:\Users\Nasteho Abdi\employee-attrition-prediction\reports\cleaning_summary.json


In [30]:
with REMOVED_COLUMNS_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    if removed_column_names:
        for column in removed_column_names:
            file.write(f"{column}\n")
    else:
        file.write("No columns were removed.\n")

print(
    "Removed-columns report saved to:",
    REMOVED_COLUMNS_PATH.resolve(),
)

Removed-columns report saved to: C:\Users\Nasteho Abdi\employee-attrition-prediction\reports\removed_columns.txt


In [31]:
validation_report.to_csv(
    VALIDATION_REPORT_PATH,
    index=False,
)

print(
    "Validation report saved to:",
    VALIDATION_REPORT_PATH.resolve(),
)

Validation report saved to: C:\Users\Nasteho Abdi\employee-attrition-prediction\reports\data_validation_report.csv


## 25. Reload and Verify the Saved Dataset

In [32]:
saved_df = pd.read_csv(CLEAN_DATA_PATH)

assert saved_df.shape == clean_df.shape
assert saved_df.columns.tolist() == clean_df.columns.tolist()
assert saved_df[TARGET_COLUMN].isin([0, 1]).all()
assert saved_df.duplicated().sum() == 0

print("Saved dataset verification passed.")
print("Saved shape:", saved_df.shape)

saved_df.head()

Saved dataset verification passed.
Saved shape: (1470, 31)


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EnvironmentSatisfaction,Gender,HourlyRate,JobInvolvement,JobLevel,JobRole,JobSatisfaction,MaritalStatus,MonthlyIncome,MonthlyRate,NumCompaniesWorked,OverTime,PercentSalaryHike,PerformanceRating,RelationshipSatisfaction,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,1,Travel_Rarely,1102,Sales,1,2,Life Sciences,2,Female,94,3,2,Sales Executive,4,Single,5993,19479,8,Yes,11,3,1,0,8,0,1,6,4,0,5
1,49,0,Travel_Frequently,279,Research & Development,8,1,Life Sciences,3,Male,61,2,2,Research Scientist,2,Married,5130,24907,1,No,23,4,4,1,10,3,3,10,7,1,7
2,37,1,Travel_Rarely,1373,Research & Development,2,2,Other,4,Male,92,2,1,Laboratory Technician,3,Single,2090,2396,6,Yes,15,3,2,0,7,3,3,0,0,0,0
3,33,0,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,4,Female,56,3,1,Research Scientist,3,Married,2909,23159,1,Yes,11,3,3,0,8,3,3,8,7,3,0
4,27,0,Travel_Rarely,591,Research & Development,2,1,Medical,1,Male,40,3,1,Laboratory Technician,2,Married,3468,16632,9,No,12,3,4,1,6,3,3,2,2,2,2


# Data Cleaning Summary

The dataset was cleaned and validated successfully.

## Completed Steps

- Loaded the original IBM employee attrition dataset
- Preserved the original data by using a working copy
- Removed duplicate rows when present
- Removed constant columns that contain no predictive information
- Removed `EmployeeNumber` because it is an identifier
- Trimmed whitespace from categorical values
- Validated important categorical domains
- Validated numerical ranges
- Checked logical relationships between work-experience columns
- Converted `Attrition` from `Yes/No` into `1/0`
- Saved the cleaned dataset and validation reports

## Generated Files

```text
data/processed/
└── clean_employee_attrition.csv

reports/
├── cleaning_summary.json
├── removed_columns.txt
└── data_validation_report.csv
```

